# ENERGIZE NILM Structured Pruning, Fine-tuning & Evaluation

This notebook supports **CNN** and **TCN** models and any PLEGMA appliance.

Pipeline:
1. **Configure** choose model, appliance and pruning ratio in one cell
2. **Baseline** load the trained checkpoint, measure cost, evaluate on test set
3. **Prune** apply global structured channel pruning (50% by default)
4. **Evaluate pruned** test-set metrics immediately after pruning
5. **Fine-tune** x-epoch recovery training on the training set
6. **Evaluate fine-tuned** test-set metrics after fine-tuning
7. **Export** save all results (Params, MACs, MB + metrics) to Excel

> Pruning functions live in `src_pytorch/pruner.py` and are imported here.

---

## Google Colab Setup
1. Upload your `ENERGIZE` folder to Google Drive
2. Run the cell below first and edit `DRIVE_PROJECT_PATH`

In [2]:
# ============================================================================
# COLAB SETUP â run this cell first
# ============================================================================
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_pruning', 'openpyxl'])

    # =========================================================================
    DRIVE_PROJECT_PATH =  '/content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL'  # <-- EDIT THIS
    # =========================================================================

    import os
    from pathlib import Path
    project_root = Path(DRIVE_PROJECT_PATH)
    if not project_root.exists():
        raise FileNotFoundError(f"Project folder not found: {project_root}")
    os.chdir(project_root)
    sys.path.insert(0, str(project_root))
    print(f"Project root: {project_root}")
else:
    import os
    from pathlib import Path
    project_root = Path(os.getcwd()).parent
    sys.path.insert(0, str(project_root))
    print(f"Running locally. Project root: {project_root}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL


## 1. Imports

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch_pruning as tp
from tqdm import tqdm
from types import SimpleNamespace

# NILM package — models, data, evaluation, pipeline helpers
from src_pytorch import (
    CNN_NILM, TCN_NILM, CNN_NILM_Seq2Seq,
    SimpleNILMDataLoader,
    set_seeds, get_device, count_parameters,
    compute_status,
    compute_metrics,
    evaluate_model,
    save_pipeline_results,
    get_appliance_params,
    get_model_config,
)
from src_pytorch.pruner import (
    count_parameters_per_layer,
    get_model_stats,
    param_ratio_to_channel_ratio,
    apply_torch_pruning,
)

set_seeds(42)
device = get_device()

print(f"PyTorch version : {torch.__version__}")
print(f"torch_pruning   : {tp.__version__}")
print(f"Device          : {device}")

Seeds set to 42
Using GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.09 GB
PyTorch version : 2.10.0+cu128
torch_pruning   : 1.6.0
Device          : cuda


In [4]:
import src_pytorch
print(dir(src_pytorch))

['CALLBACKS', 'CNN_NILM', 'CNN_NILM_Seq2Seq', 'DATASET_CONFIGS', 'DATASET_SPLITS', 'DataLoaderNILM', 'EarlyStopping', 'MODEL_CONFIGS', 'ModelCheckpoint', 'NILMDataset', 'PLEGMA_PARAMS', 'REFIT_PARAMS', 'SimpleNILMDataLoader', 'SimpleTester', 'TCN_NILM', 'TRAINING', 'Trainer', 'TrainingHistory', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'apply_torch_pruning', 'apply_unstructured_pruning', 'build_nilm_model', 'compute_metrics', 'compute_status', 'config', 'count_ops_and_params', 'count_parameters', 'count_parameters_per_layer', 'create_experiment_directories', 'data_loader', 'evaluate_model', 'evaluator', 'get_appliance_params', 'get_data_loader', 'get_dataset_config', 'get_dataset_split', 'get_device', 'get_model', 'get_model_config', 'get_model_sparsity', 'get_model_stats', 'get_prunable_parameters', 'load_checkpoint', 'load_model', 'models', 'param_ratio_to_channel_ratio', 'pipeline',

## 2. Configuration

**Edit only this cell.** Everything else adapts automatically.

In [5]:
# ============================================================================
# USER CONFIGURATION
# ============================================================================
MODEL_NAME     = 'cnn_seq2seq'      # 'cnn'  |  'cnn_seq2seq'  |  'tcn'
APPLIANCE_NAME = 'boiler'   # 'boiler'  |  'ac_1'  |  'washing_machine'
PRUNING_RATIO  = 0.5        # target *parameter* reduction  (0.0 – 1.0)
FINETUNE_LR    = 1e-4       # Adam learning rate for 1-epoch fine-tuning
DATASET_NAME   = 'plegma'

# ============================================================================
# AUTO-DERIVED — do not edit below this line
# ============================================================================
# Model config from src_pytorch (single source of truth)
_mcfg = get_model_config(MODEL_NAME)
cfg = {
    'window'          : _mcfg['input_window_length'],
    'batch_size'      : _mcfg['batch_size'],
    'depth'           : _mcfg.get('depth', 9),
    'filters'         : _mcfg.get('nb_filters'),
    'dropout'         : _mcfg.get('dropout', 0.2),
    'stacks'          : _mcfg.get('stacks', 1),
    # Protect the correct output layer: CNN Seq2Point → 1, CNN Seq2Seq / TCN → window
    'args_window_size': 1 if MODEL_NAME == 'cnn' else _mcfg['input_window_length'],
}

# Appliance config from src_pytorch (single source of truth — synced with config.py)
app_cfg = get_appliance_params(DATASET_NAME, APPLIANCE_NAME)

WINDOW                 = cfg['window']
BATCH_SIZE             = cfg['batch_size']
THRESHOLD              = app_cfg['threshold']
CUTOFF                 = app_cfg['cutoff']
MIN_ON                 = app_cfg.get('min_on')
MIN_OFF                = app_cfg.get('min_off')
MIN_COMMITTED_DURATION = app_cfg.get('min_committed_duration')
pct                    = int(PRUNING_RATIO * 100)

# Convert user-facing parameter ratio → internal channel ratio for MetaPruner.
_channel_ratio = param_ratio_to_channel_ratio(PRUNING_RATIO)

# ── Output directories ───────────────────────────────────────────────────────
OUT_DIR     = project_root / 'outputs' / f'{MODEL_NAME}_{APPLIANCE_NAME}'
MODELS_DIR  = OUT_DIR / 'models'
PREDS_DIR   = OUT_DIR / 'predictions'
METRICS_DIR = OUT_DIR / 'metrics'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PREDS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH  = OUT_DIR / 'checkpoint' / 'model.pt'
DATA_DIR   = Path('/content/drive/MyDrive/Colab Notebooks/ENERGIZE') / 'data' / 'processed' / DATASET_NAME / APPLIANCE_NAME
EXCEL_PATH = OUT_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_comparative_results.xlsx'

print(f"Model                : {MODEL_NAME.upper()}")
print(f"Appliance            : {APPLIANCE_NAME}")
print(f"Target param removal : {pct}%  →  channel ratio: {_channel_ratio:.4f}")
print(f"Fine-tune LR         : {FINETUNE_LR}")
print(f"Window length        : {WINDOW}")
print(f"Threshold            : {THRESHOLD} W  |  Cutoff: {CUTOFF} W")
print(f"Min ON               : {MIN_ON} samples  |  Min OFF: {MIN_OFF} samples")
if MIN_COMMITTED_DURATION:
    print(f"Min committed dur.   : {MIN_COMMITTED_DURATION} samples")
print(f"Checkpoint           : {CKPT_PATH}  ({'found' if CKPT_PATH.exists() else 'NOT FOUND'})")
print(f"Models dir           : {MODELS_DIR}")
print(f"Predictions dir      : {PREDS_DIR}")
print(f"Metrics dir          : {METRICS_DIR}")
print(f"Excel                : {EXCEL_PATH}")

Model                : CNN_SEQ2SEQ
Appliance            : boiler
Target param removal : 50%  →  channel ratio: 0.2929
Fine-tune LR         : 0.0001
Window length        : 299
Threshold            : 800 W  |  Cutoff: 4000 W
Min ON               : 30 samples  |  Min OFF: 6 samples
Checkpoint           : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/checkpoint/model.pt  (found)
Models dir           : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/models
Predictions dir      : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/predictions
Metrics dir          : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/metrics
Excel                : /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/cnn_seq2seq_boiler_comparative_res

In [6]:
data_loader = SimpleNILMDataLoader(
    data_dir=str(DATA_DIR),
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    input_window_length=WINDOW,
    train=True,
    num_workers=0
)

print(f"Train batches : {len(data_loader.train)}")
print(f"Val   batches : {len(data_loader.val)}")
print(f"Test  batches : {len(data_loader.test)}")

Train batches : 46335
Val   batches : 9
Test  batches : 11


## 4. Helper Model Factory

A single function that builds and loads the correct architecture from a checkpoint.

In [7]:
def build_model(model_name: str, cfg: dict, ckpt_path, device) -> nn.Module:
    """Instantiate and load a NILM model from a checkpoint."""
    if model_name == 'cnn':
        model = CNN_NILM(input_window_length=cfg['window'])
    elif model_name == 'cnn_seq2seq':
        model = CNN_NILM_Seq2Seq(input_window_length=cfg['window'])
    elif model_name == 'tcn':
        model = TCN_NILM(
            input_window_length=cfg['window'],
            depth=cfg['depth'],
            nb_filters=cfg['filters'],
            dropout=cfg['dropout'],
            stacks=cfg['stacks'],
        )
    else:
        raise ValueError(f"Unknown model: {model_name}. Choose 'cnn', 'cnn_seq2seq', or 'tcn'.")
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return model.to(device).eval()


def evaluate_and_save(model, label, data_loader, model_name, cutoff, threshold,
                      device, input_window_length, preds_dir, prefix,
                      min_on=None, min_off=None, min_committed_duration=None):
    """
    Run test-set inference via evaluate_model, save predictions CSV, return metrics.
    """
    metrics, gt, pred, gt_status, pred_status = evaluate_model(
        model=model,
        data_loader=data_loader,
        model_name=model_name,
        cutoff=cutoff,
        threshold=threshold,
        device=device,
        input_window_length=input_window_length,
        min_on=min_on,
        min_off=min_off,
        min_committed_duration=min_committed_duration,
    )

    pred_data = {'ground_truth_W': gt, 'prediction_W': pred}
    if gt_status is not None:
        pred_data['ground_truth_status'] = gt_status.astype(int)
        pred_data['predicted_status']    = pred_status.astype(int)

    csv_path = preds_dir / f'{prefix}_{label}_preds.csv'
    pd.DataFrame(pred_data).to_csv(csv_path, index=False)
    print(f"  Predictions saved : {csv_path.name}  (columns: {list(pred_data.keys())})")
    return metrics


print("build_model() and evaluate_and_save() defined.")

build_model() and evaluate_and_save() defined.


## 5. Baseline Load & Evaluate

In [8]:
# Load baseline model
baseline_model = build_model(MODEL_NAME, cfg, CKPT_PATH, device)
dummy_input    = torch.randn(1, WINDOW).to(device)

baseline_params, baseline_macs, baseline_mb = get_model_stats(baseline_model, dummy_input)

print(f"Model      : {MODEL_NAME.upper()}")
print(f"Parameters : {baseline_params:,}")
print(f"MACs       : {baseline_macs:,}")
print(f"Size       : {baseline_mb:.3f} MB")

Model      : CNN_SEQ2SEQ
Parameters : 14,168,899
MACs       : 24,484,743.0
Size       : 54.050 MB


In [9]:
_prefix = f'{MODEL_NAME}_{APPLIANCE_NAME}'

print(f"Evaluating {MODEL_NAME.upper()} baseline on test set...")
baseline_metrics = evaluate_and_save(
    model=baseline_model,
    label='baseline',
    data_loader=data_loader,
    model_name=MODEL_NAME,
    cutoff=CUTOFF,
    threshold=THRESHOLD,
    device=device,
    input_window_length=WINDOW,
    preds_dir=PREDS_DIR,
    prefix=_prefix,
    min_on=MIN_ON,
    min_off=MIN_OFF,
    min_committed_duration=MIN_COMMITTED_DURATION,
)

print(f"\nBaseline Results:")
for k, v in baseline_metrics.items():
    print(f"  {k:<25}: {v:.4f}")

Evaluating CNN_SEQ2SEQ baseline on test set...


Inference: 100%|██████████| 11/11 [00:00<00:00, 51.32it/s]


  Predictions saved : cnn_seq2seq_boiler_baseline_preds.csv  (columns: ['ground_truth_W', 'prediction_W', 'ground_truth_status', 'predicted_status'])

Baseline Results:
  mae                      : 7.9734
  accuracy                 : 0.9984
  tp                       : 27457.0000
  tn                       : 1596720.0000
  fp                       : 2253.0000
  fn                       : 429.0000
  total_gt_energy_wh       : 22142.4775
  total_pred_energy_wh     : 20930.1406
  energy_error_percent     : 5.4752
  precision_complex        : 0.9298
  recall_complex           : 0.9843
  f1_complex               : 0.9562


## 6. Prune & Evaluate

In [10]:
# Reload a fresh instance â pruning is irreversible
pruned_model = build_model(MODEL_NAME, cfg, CKPT_PATH, device)
pruning_args = SimpleNamespace(window_size=cfg['args_window_size'])

pruned_model, _ = apply_torch_pruning(
    model=pruned_model,
    args=pruning_args,
    inputs=dummy_input,
    pruning_ratio=PRUNING_RATIO,
)

Baseline model  — MACs: 24,484,743.0  |  Params: 14,168,899
Pruned model    — MACs: 12,123,437.0  |  Params: 7,077,534
MACs reduction  : 50.5%
Param reduction : 50.0%
Output shape    : torch.Size([1, 299, 1])


In [11]:
# Model cost after pruning
pruned_params, pruned_macs, pruned_mb = get_model_stats(pruned_model, dummy_input)

print(f"Parameters : {pruned_params:,}  ({(1 - pruned_params/baseline_params)*100:.1f}% reduction)")
print(f"MACs       : {pruned_macs:,}  ({(1 - pruned_macs/baseline_macs)*100:.1f}% reduction)")
print(f"Size       : {pruned_mb:.3f} MB  ({(1 - pruned_mb/baseline_mb)*100:.1f}% reduction)")

print("\nPer-layer parameter counts (after pruning):")
for name, cnt in count_parameters_per_layer(pruned_model).items():
    print(f"  {name:<60} {cnt:>10,}")

Parameters : 7,077,534  (50.0% reduction)
MACs       : 12,123,437.0  (50.5% reduction)
Size       : 26.999 MB  (50.0% reduction)

Per-layer parameter counts (after pruning):
  network.0                                                           253
  network.2                                                         3,515
  network.4                                                         3,680
  network.6                                                         5,152
  network.9                                                         5,635
  network.13                                                    6,842,524
  network.16                                                      216,775


In [12]:
print(f"Evaluating pruned {MODEL_NAME.upper()} on test set...")
pruned_metrics = evaluate_and_save(
    model=pruned_model,
    label=f'pruned_{pct}pct',
    data_loader=data_loader,
    model_name=MODEL_NAME,
    cutoff=CUTOFF,
    threshold=THRESHOLD,
    device=device,
    input_window_length=WINDOW,
    preds_dir=PREDS_DIR,
    prefix=_prefix,
    min_on=MIN_ON,
    min_off=MIN_OFF,
    min_committed_duration=MIN_COMMITTED_DURATION,
)

print(f"\nPruned Results:")
for k, v in pruned_metrics.items():
    print(f"  {k:<25}: {v:.4f}")

Evaluating pruned CNN_SEQ2SEQ on test set...


Inference: 100%|██████████| 11/11 [00:00<00:00, 79.73it/s]


  Predictions saved : cnn_seq2seq_boiler_pruned_50pct_preds.csv  (columns: ['ground_truth_W', 'prediction_W', 'ground_truth_status', 'predicted_status'])

Pruned Results:
  mae                      : 48.9981
  accuracy                 : 0.9829
  tp                       : 0.0000
  tn                       : 1598973.0000
  fp                       : 0.0000
  fn                       : 27886.0000
  total_gt_energy_wh       : 22142.4775
  total_pred_energy_wh     : 0.0000
  energy_error_percent     : 100.0000
  precision_complex        : 0.0000
  recall_complex           : 0.0000
  f1_complex               : 0.0000


In [13]:
pruned_ckpt = MODELS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct.pt'
torch.save(pruned_model.state_dict(), pruned_ckpt)
print(f"Pruned checkpoint saved: {pruned_ckpt}")

Pruned checkpoint saved: /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/models/cnn_seq2seq_boiler_pruned_50pct.pt


## 7. Fine-tune

`FINETUNE_EPOCHS` epochs of MSE training on the training set with Adam at `FINETUNE_LR`.
The pruned model structure is preserved — no re-growing of pruned channels.

In [14]:
FINETUNE_EPOCHS =1

In [15]:
optimizer = torch.optim.Adam(pruned_model.parameters(), lr=FINETUNE_LR)
loss_fn   = nn.MSELoss()

for epoch in range(1, FINETUNE_EPOCHS + 1):
    pruned_model.train()
    total_loss = 0.0
    total_mae  = 0.0
    n_batches  = 0

    for batch_x, batch_y in tqdm(data_loader.train, desc=f"Fine-tuning epoch {epoch}/{FINETUNE_EPOCHS}"):
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        outputs = pruned_model(batch_x)

        # Align target shape to model output
        # CNN (Seq2Point)   : output (B, 1),          target (B,)         → (B, 1)
        # CNN Seq2Seq / TCN : output (B, seq_len, 1), target (B, seq_len) → (B, seq_len, 1)
        if MODEL_NAME == 'cnn' and batch_y.dim() == 1:
            batch_y = batch_y.unsqueeze(1)
        elif MODEL_NAME in ('cnn_seq2seq', 'tcn') and batch_y.dim() == 2:
            batch_y = batch_y.unsqueeze(-1)

        loss = loss_fn(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_mae  += torch.mean(torch.abs(outputs.detach() - batch_y)).item()
        n_batches  += 1

    print(f"Epoch {epoch}/{FINETUNE_EPOCHS}  avg MSE loss: {total_loss/n_batches:.6f}  "
          f"avg MAE: {total_mae/n_batches:.6f}")

pruned_model.eval()
print("\nFine-tuning complete.")

Fine-tuning epoch 1/1: 100%|██████████| 46335/46335 [12:34<00:00, 61.43it/s]

Epoch 1/1  avg MSE loss: 0.000557  avg MAE: 0.003470

Fine-tuning complete.


In [16]:
finetuned_ckpt = MODELS_DIR / f'{MODEL_NAME}_{APPLIANCE_NAME}_pruned_{pct}pct_finetuned.pt'
torch.save(pruned_model.state_dict(), finetuned_ckpt)
print(f"Fine-tuned checkpoint saved: {finetuned_ckpt}")

Fine-tuned checkpoint saved: /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/models/cnn_seq2seq_boiler_pruned_50pct_finetuned.pt


## 8. Evaluate Fine-tuned Model

Architecture (and therefore Params / MACs / MB) is unchanged by fine-tuning.

In [17]:
finetuned_params, finetuned_macs, finetuned_mb = get_model_stats(pruned_model, dummy_input)

print(f"Evaluating fine-tuned {MODEL_NAME.upper()} on test set...")
finetuned_metrics = evaluate_and_save(
    model=pruned_model,
    label=f'pruned_{pct}pct_finetuned',
    data_loader=data_loader,
    model_name=MODEL_NAME,
    cutoff=CUTOFF,
    threshold=THRESHOLD,
    device=device,
    input_window_length=WINDOW,
    preds_dir=PREDS_DIR,
    prefix=_prefix,
    min_on=MIN_ON,
    min_off=MIN_OFF,
    min_committed_duration=MIN_COMMITTED_DURATION,
)

print(f"\nFine-tuned Results:")
for k, v in finetuned_metrics.items():
    print(f"  {k:<25}: {v:.4f}")

Evaluating fine-tuned CNN_SEQ2SEQ on test set...


Inference: 100%|██████████| 11/11 [00:00<00:00, 86.15it/s]


  Predictions saved : cnn_seq2seq_boiler_pruned_50pct_finetuned_preds.csv  (columns: ['ground_truth_W', 'prediction_W', 'ground_truth_status', 'predicted_status'])

Fine-tuned Results:
  mae                      : 8.0886
  accuracy                 : 0.9975
  tp                       : 27538.0000
  tn                       : 1595294.0000
  fp                       : 3679.0000
  fn                       : 348.0000
  total_gt_energy_wh       : 22142.4775
  total_pred_energy_wh     : 22043.1680
  energy_error_percent     : 0.4485
  precision_complex        : 0.8852
  recall_complex           : 0.9851
  f1_complex               : 0.9325


## 9. Export Results to Excel

In [18]:
def make_row(label, pruning_pct, params, macs, mb, metrics):
    return {
        'Model'             : label,
        'Architecture'      : MODEL_NAME.upper(),
        'Appliance'         : APPLIANCE_NAME,
        'Pruning_Ratio_%'   : pruning_pct,
        'Params'            : params,
        'MACs'              : macs,
        'MB'                : mb,
        'MAE'               : round(metrics['mae'],                          4),
        'F1_Complex'        : round(metrics['f1_complex'],        4) if 'f1_complex'        in metrics else '',
        'Precision_Complex' : round(metrics['precision_complex'], 4) if 'precision_complex' in metrics else '',
        'Recall_Complex'    : round(metrics['recall_complex'],    4) if 'recall_complex'    in metrics else '',
        'Accuracy'          : round(metrics['accuracy'],                     4),
        'GT_Energy_Wh'      : round(metrics['total_gt_energy_wh'],           2),
        'Pred_Energy_Wh'    : round(metrics['total_pred_energy_wh'],         2),
    }


results_df = pd.DataFrame([
    make_row(f'{MODEL_NAME.upper()} Baseline',
             0, baseline_params, baseline_macs, baseline_mb, baseline_metrics),
    make_row(f'{MODEL_NAME.upper()} Pruned {pct}%',
             pct, pruned_params, pruned_macs, pruned_mb, pruned_metrics),
    make_row(f'{MODEL_NAME.upper()} Pruned {pct}% + Fine-tuned {FINETUNE_EPOCHS}ep',
             pct, finetuned_params, finetuned_macs, finetuned_mb, finetuned_metrics),
])

# Load existing Excel and upsert rows (so notebook 04 entries are preserved)
if EXCEL_PATH.exists():
    existing = pd.read_excel(EXCEL_PATH)
    known    = set(results_df['Model'])
    updated  = pd.concat(
        [existing[~existing['Model'].isin(known)], results_df],
        ignore_index=True,
    )
else:
    updated = results_df

updated.to_excel(EXCEL_PATH, index=False)
print(f"Results saved to: {EXCEL_PATH}")
updated

Results saved to: /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/cnn_seq2seq_boiler_comparative_results.xlsx


,Model,Architecture,Appliance,Pruning_Ratio_%,Params,MACs,MB,MAE,F1_Complex,Precision_Complex,Recall_Complex,Accuracy,GT_Energy_Wh,Pred_Energy_Wh
0,CNN_SEQ2SEQ Baseline,CNN_SEQ2SEQ,boiler,0,14168899,24484743.0,54.050,7.9734,0.9562,0.9298,0.9843,0.9984,22142.48,20930.140625
1,CNN_SEQ2SEQ Pruned 50%,CNN_SEQ2SEQ,boiler,50,7077534,12123437.0,26.999,48.9981,0.0000,0.0000,0.0000,0.9829,22142.48,0.000000
2,CNN_SEQ2SEQ Pruned 50% + Fine-tuned 1ep,CNN_SEQ2SEQ,boiler,50,7077534,12123437.0,26.999,8.0886,0.9325,0.8852,0.9851,0.9975,22142.48,22043.169922


In [19]:
# Save pipeline metrics CSV to metrics/ folder (mirrors main.py behaviour)
save_pipeline_results(
    rows=[
        {**baseline_metrics,  'label': f'{MODEL_NAME.upper()} Baseline'},
        {**pruned_metrics,    'label': f'{MODEL_NAME.upper()} Pruned {pct}%'},
        {**finetuned_metrics, 'label': f'{MODEL_NAME.upper()} Pruned {pct}% + Fine-tuned 1ep'},
    ],
    output_dir=METRICS_DIR,
    appliance=APPLIANCE_NAME,
    model_name=MODEL_NAME,
)



  Pipeline results saved to: /content/drive/MyDrive/Colab Notebooks/ENERGIZE_COMPRESSION_FRAMEWORK FINAL/outputs/cnn_seq2seq_boiler/metrics/boiler_cnn_seq2seq_pipeline_results.csv


# Save pipeline metrics CSV to metrics/ folder (mirrors main.py behaviour)


In [ ]:
save_pipeline_results(
    rows=[
        {**baseline_metrics,  'label': f'{MODEL_NAME.upper()} Baseline'},
        {**pruned_metrics,    'label': f'{MODEL_NAME.upper()} Pruned {pct}%'},
        {**finetuned_metrics, 'label': f'{MODEL_NAME.upper()} Pruned {pct}% + Fine-tuned {FINETUNE_EPOCHS}ep'},
    ],
    output_dir=METRICS_DIR,
    appliance=APPLIANCE_NAME,
    model_name=MODEL_NAME,
)

In [ ]:
C, W3 = 22, 16
SEP   = "=" * (C + W3 * 3 + 3)
sep   = "-" * (C + W3 * 3 + 3)
hfmt  = f"{{:<{C}}} {{:>{W3}}} {{:>{W3}}} {{:>{W3}}}"
rfmt  = f"{{:<{C}}} {{:>{W3}}} {{:>{W3}}} {{:>{W3}}}"

col_baseline  = f"{MODEL_NAME.upper()} Baseline"
col_pruned    = f"Pruned {pct}%"
col_finetuned = f"Pruned+FT {FINETUNE_EPOCHS}ep"

print(SEP)
print(f"PRUNING SUMMARY  |  {MODEL_NAME.upper()}  |  {APPLIANCE_NAME}  |  pruning={pct}%  |  LR={FINETUNE_LR}")
print(SEP)
print(hfmt.format('Metric', col_baseline, col_pruned, col_finetuned))
print(sep)

def rrow(label, b, p, f):
    print(rfmt.format(label, b, p, f))

rrow('Params',        f"{baseline_params:,}",   f"{pruned_params:,}",   f"{finetuned_params:,}")
rrow('MACs',          f"{baseline_macs:,}",     f"{pruned_macs:,}",     f"{finetuned_macs:,}")
rrow('MB',            f"{baseline_mb:.3f}",     f"{pruned_mb:.3f}",     f"{finetuned_mb:.3f}")
print(sep)
rrow('MAE (W)',       f"{baseline_metrics['mae']:.4f}",
                      f"{pruned_metrics['mae']:.4f}",
                      f"{finetuned_metrics['mae']:.4f}")
if 'f1_complex' in baseline_metrics:
    rrow('F1 Complex',    f"{baseline_metrics['f1_complex']:.4f}",
                          f"{pruned_metrics['f1_complex']:.4f}",
                          f"{finetuned_metrics['f1_complex']:.4f}")
    rrow('Prec Complex',  f"{baseline_metrics['precision_complex']:.4f}",
                          f"{pruned_metrics['precision_complex']:.4f}",
                          f"{finetuned_metrics['precision_complex']:.4f}")
    rrow('Rec Complex',   f"{baseline_metrics['recall_complex']:.4f}",
                          f"{pruned_metrics['recall_complex']:.4f}",
                          f"{finetuned_metrics['recall_complex']:.4f}")
rrow('Accuracy',      f"{baseline_metrics['accuracy']:.4f}",
                      f"{pruned_metrics['accuracy']:.4f}",
                      f"{finetuned_metrics['accuracy']:.4f}")
rrow('GT Energy Wh',  f"{baseline_metrics['total_gt_energy_wh']:.2f}",
                      f"{pruned_metrics['total_gt_energy_wh']:.2f}",
                      f"{finetuned_metrics['total_gt_energy_wh']:.2f}")
rrow('Pred Energy Wh',f"{baseline_metrics['total_pred_energy_wh']:.2f}",
                      f"{pruned_metrics['total_pred_energy_wh']:.2f}",
                      f"{finetuned_metrics['total_pred_energy_wh']:.2f}")
print(SEP)
print(f"Excel  : {EXCEL_PATH}")
print(f"Models : {MODELS_DIR}")
print(f"  {pruned_ckpt.name}")
print(f"  {finetuned_ckpt.name}")
print(f"Preds  : {PREDS_DIR}")
print(SEP)